# Training Sharada on a free T4

One session trains one model. `sharada-base` takes about an hour, `sharada-large` about three, so
large needs either a session that is left alone or one restart — the run checkpoints every 500
steps and picks up where it stopped.

**Before running:** Settings → Accelerator → *GPU T4 x2* (one card is used), Internet **on**. To
upload the result, add your Hub token as a secret named `HF_TOKEN` (Add-ons → Secrets).

**To continue an interrupted run:** add this notebook's previous version as an input dataset
(*+ Add Input → Your Work → Notebook Output*) and the checkpoint cell below will find it.

## The code and what it needs

In [ ]:
!pip install -q 'transformers==5.12.1' 'datasets>=4' safetensors huggingface-hub
!git clone -q https://github.com/LenaBarretta/sharada /kaggle/working/sharada
!pip install -q -e /kaggle/working/sharada --no-deps

import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

## Pick the model

`base` is the default to fine-tune; `large` is a few points better and about 2.5× the time.

In [ ]:
WHICH = 'base'                      # 'base' or 'large'

ENCODER = f'answerdotai/ModernBERT-{WHICH}'
OUT = f'/kaggle/working/{WHICH}'
STEPS = 12_000
BATCH = 16 if WHICH == 'base' else 8
LR = 3e-5 if WHICH == 'base' else 2e-5
PUSH_TO = None                      # e.g. f'LenaBarretta/sharada-{WHICH}', or leave None
ENCODER, OUT, BATCH

## Carry on from an earlier session, if there is one

In [ ]:
import pathlib, shutil

found = sorted(pathlib.Path('/kaggle/input').glob(f'*/{WHICH}/checkpoint.pt'))
if found and not pathlib.Path(OUT, 'checkpoint.pt').exists():
    pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
    shutil.copy(found[-1], pathlib.Path(OUT, 'checkpoint.pt'))
    print('resuming from', found[-1])
else:
    print('starting from scratch' if not found else 'a checkpoint is already in place')

## The run

It prints the mix it managed to load (a dataset that has moved on the Hub is skipped, not fatal),
then a line every hundred steps with the loss and an estimate of what is left, and every thousand
steps it measures accuracy, log loss and calibration error — including on the label sets it is
never trained on, which is the number worth watching.

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!cd /kaggle/working/sharada && python training/run.py \
    --encoder {ENCODER} --out {OUT} --steps {STEPS} --batch-size {BATCH} --lr {LR} \
    --eval-every 1000 --checkpoint-every 500

## What came out

In [ ]:
import json

report = json.load(open(f'{OUT}/report.json'))
print(f"{report['training_examples']:,} examples, {report['label_sets']} label sets,"
      f" {report['steps']:,} steps")
print('overall:', {k: round(v, 4) for k, v in report['overall'].items() if isinstance(v, float)})
print('one decision:', round(report['latency']['ms_per_question'], 1), 'ms\n')
for task, row in sorted(report['per_task'].items(), key=lambda kv: -kv[1]['accuracy']):
    mark = '    ' if row['trained_on'] else 'new '
    print(f"{mark}{task:20s} {row['options']:3d} options  accuracy {row['accuracy']:.3f}"
          f"  ece {row['calibration_error']:.3f}  T {row['temperature']}")

In [ ]:
import matplotlib.pyplot as plt

history = report['history']
steps = [h['step'] for h in history]
figure, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for axis, key, title in zip(axes, ['accuracy', 'log_loss', 'calibration_error'],
                            ['accuracy', 'log loss', 'calibration error']):
    axis.plot(steps, [h[key] for h in history], marker='o')
    axis.set_title(title); axis.set_xlabel('step'); axis.grid(alpha=.3)
axes[0].plot(steps, [h['holdout_accuracy'] for h in history], marker='o', label='unseen label sets')
axes[0].legend()
plt.tight_layout(); plt.show()

## Try it before trusting it

In [ ]:
from sharada import DecisionModel

model = DecisionModel.from_pretrained(OUT)
ticket = "My card still hasn't arrived and I ordered it two weeks ago. Nobody answers the chat."

for question, options, kind in [
    ('Which team should handle this?',
     ['billing', 'card delivery', 'technical support', 'account closure'], 'choice'),
    ('How urgent is this?', ['can wait', 'this week', 'today', 'right now'], 'scale'),
    ('Is the customer angry?', ['no', 'yes'], 'binary'),
]:
    d = model.decide(ticket, question, options, kind=kind)
    print(f'{question:40s} {d.answer:22s} {d.confidence:.2f}')

## Upload it

`publish.py` writes the model card out of `report.json`, so what is on the Hub is what was
measured here, and uploads everything except the training checkpoint.

In [ ]:
if PUSH_TO:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    !cd /kaggle/working/sharada && python training/publish.py {OUT} --repo {PUSH_TO}
else:
    print('PUSH_TO is None — set it above to upload')

The checkpoint stays in `/kaggle/working`, so this notebook's output is what a next session
resumes from. Delete it once the model is on the Hub: it is the optimiser state, not the model.